# Notebook 16 – Cross Validation

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity` and `UnitPrice`.

## 1. Load & Prepare Data
We load the CSV, take a smaller random sample (5,000 rows) so everything runs quickly, and pick our features (`X`) and target (`y`).

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
df = df.sample(5000, random_state=42).reset_index(drop=True)  
X = df[['Quantity', 'UnitPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X.head()

,Quantity,UnitPrice
0,24,0.85
1,4,6.95
2,4,0.65
3,3,1.95
4,2,9.95


## 2. Why Cross Validation?

If we test our model on just **one** train/test split, the score we get depends a lot on *which* rows happened to land in the test set. We might get lucky (score looks great) or unlucky (score looks bad), and we would not really know which.

**Cross validation** fixes this by testing the model on several different splits and averaging the results, giving a more trustworthy picture of how well the model actually performs.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)
print("Single split accuracy:", accuracy_score(y_test, model.predict(X_test)))

Single split accuracy: 0.903


## 3. K-Fold Cross Validation

The data is split into **K equal parts (folds)**. The model is trained on K-1 folds and tested on the remaining fold. This repeats K times, so every row gets used for testing exactly once. The final score is the average across all K rounds.

In [4]:
from sklearn.model_selection import KFold, cross_val_score
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LogisticRegression(), X, y, cv=kf)
print("K-Fold scores:", scores)
print("Average:", scores.mean())

K-Fold scores: [0.903 0.916 0.908 0.904 0.899]
Average: 0.9059999999999999


## 4. Stratified K-Fold

Regular K-Fold splits randomly, so one fold might end up with very few UK orders by chance. **Stratified K-Fold** makes sure each fold keeps roughly the same UK vs non-UK ratio as the full dataset — important for imbalanced classes like ours (most orders are UK).

In [5]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(LogisticRegression(), X, y, cv=skf)
print("Stratified scores:", scores_strat)
print("Average:", scores_strat.mean())

Stratified scores: [0.906 0.907 0.904 0.906 0.906]
Average: 0.9057999999999999


## 5. Leave-One-Out (LOO)

An extreme version of K-Fold where **K equals the number of rows** — each fold tests on just a single data point and trains on all the rest. It gives a very thorough estimate but is extremely slow on large datasets, so here we only run it on 100 rows.

In [6]:
from sklearn.model_selection import LeaveOneOut
X_small, y_small = X[:100], y[:100]
loo = LeaveOneOut()
scores_loo = cross_val_score(LogisticRegression(), X_small, y_small, cv=loo)
print("LOO average accuracy:", scores_loo.mean())

LOO average accuracy: 0.9


## 6. Training vs Validation Performance

Comparing accuracy on the **training set** vs the **validation/test set** tells us if the model is overfitting. If training accuracy is much higher than validation accuracy, the model has memorized the training data instead of learning general patterns.

In [7]:
train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc = accuracy_score(y_test, model.predict(X_test))
print("Train accuracy:", train_acc)
print("Validation accuracy:", val_acc)

Train accuracy: 0.90675
Validation accuracy: 0.903


## 7. Model Stability

Beyond the average score, we should check the **spread (standard deviation)** of scores across folds. A small spread means the model performs consistently no matter which data it sees — a sign of a stable, reliable model. A large spread means performance depends heavily on the specific split.

In [8]:
print("K-Fold std dev:", scores.std())
print("Stratified std dev:", scores_strat.std())

K-Fold std dev: 0.005761944116355179
Stratified std dev: 0.000979795897113272


## 8. Comparing All Strategies
A quick side-by-side summary of the average accuracy from each method.

In [9]:
results = pd.DataFrame({
    'Method': ['K-Fold', 'Stratified K-Fold', 'Leave-One-Out'],
    'Mean Accuracy': [scores.mean(), scores_strat.mean(), scores_loo.mean()]
})
results

,Method,Mean Accuracy
0,K-Fold,0.9060
1,Stratified K-Fold,0.9058
2,Leave-One-Out,0.9000


## Conclusion
- **K-Fold** and **Stratified K-Fold** gave similar, stable results, since our classes aren't too imbalanced in the sample.
- **Leave-One-Out** is very thorough but far too slow to use on large datasets.
- **Stratified K-Fold** is generally the safest default choice for classification tasks.